In [2]:
import sys
!{sys.executable} -m pip install pandas matplotlib plotly

  Using cached pandas-2.3.3-cp39-cp39-macosx_10_9_x86_64.whl (11.6 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 2.9 MB/s eta 0:00:0000:0100:01m
  Using cached plotly-6.6.0-py3-none-any.whl (9.9 MB)
  Using cached numpy-2.0.2-cp39-cp39-macosx_10_9_x86_64.whl (21.2 MB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl (510 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 KB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 3.0 MB/s eta 0:00:0000:0100:01
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
  Using cached pillow-11.3.0-cp39-cp39-macosx_10_10_x86_64.whl (5.3 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.5/265.5 KB 3.7 MB/s eta 0:00:00a 0:00:01
  Using cached narwhals-2.18.0-py3-none-any.whl (444 kB)
You should consider upgrading via the '/Users/ilakia/berlin-rental-analysis/venv/bin/python -m 

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/Users/ilakia/berlin-rental-analysis/processed/listings_clean.csv')
neighbourhood_summary = pd.read_csv('/Users/ilakia/berlin-rental-analysis/processed/neighbourhood_summary.csv')

print(df.shape)
df.head()

(9220, 23)


,id,name,neighbourhood_cleansed,latitude,longitude,room_type,accommodates,bedrooms,beds,price,...,review_scores_rating,review_scores_cleanliness,review_scores_location,host_is_superhost,host_response_rate,host_acceptance_rate,availability_365,reviews_per_month,calculated_host_listings_count,bathrooms
0,3176,Fabulous Flat in great Location,Prenzlauer Berg Südwest,52.53471,13.41810,Entire home/apt,2,1.0,2.0,105.0,...,4.63,4.52,4.92,False,100.0,83.0,140,0.76,1,1.0
1,9991,Geourgeous flat - outstanding views,Prenzlauer Berg Südwest,52.53269,13.41805,Entire home/apt,7,4.0,4.0,135.0,...,5.00,5.00,4.86,False,20.0,0.0,241,0.06,1,2.5
2,14325,Studio Apartment in Prenzlauer Berg,Prenzlauer Berg Nordwest,52.54813,13.40366,Entire home/apt,1,0.0,1.0,75.0,...,4.68,4.85,4.60,False,80.0,22.0,168,0.14,4,1.0
3,17904,Beautiful Kreuzberg studio - 3 months minimum,Reuterstraße,52.49419,13.42166,Entire home/apt,2,0.0,1.0,32.0,...,4.77,4.71,4.88,False,100.0,95.0,72,1.57,1,1.0
4,20858,Designer Loft in Berlin Mitte,Prenzlauer Berg Südwest,52.53711,13.40888,Entire home/apt,4,2.0,2.0,202.0,...,4.48,4.65,4.91,False,100.0,96.0,253,0.89,1,1.0


In [4]:
# Average price by room type
room_type_analysis = df.groupby('room_type').agg(
    avg_price=('price', 'mean'),
    median_price=('price', 'median'),
    count=('id', 'count')
).round(2).sort_values('avg_price', ascending=False)

print(room_type_analysis)

                 avg_price  median_price  count
room_type                                      
Hotel room          210.79         188.0     89
Entire home/apt     144.02         119.0   6777
Private room         85.98          68.0   2258
Shared room          48.21          38.0     96


In [5]:
# Superhost vs non-superhost
superhost_analysis = df.groupby('host_is_superhost').agg(
    avg_price=('price', 'mean'),
    avg_rating=('review_scores_rating', 'mean'),
    count=('id', 'count')
).round(2)

print(superhost_analysis)

                   avg_price  avg_rating  count
host_is_superhost                              
False                 123.88        4.66   5975
True                  139.79        4.86   3105


In [6]:
# Best value neighbourhoods - high rating, lower price
# Filter to neighbourhoods with at least 20 listings for reliability
value_analysis = neighbourhood_summary[neighbourhood_summary['listing_count'] >= 20].copy()

# Create a value score - rating divided by price (higher = better value)
value_analysis['value_score'] = (value_analysis['avg_rating'] / value_analysis['avg_price'] * 100).round(3)

# Best value
print("TOP 10 BEST VALUE NEIGHBOURHOODS:")
print(value_analysis.sort_values('value_score', ascending=False)[
    ['neighbourhood_cleansed', 'avg_price', 'avg_rating', 'listing_count', 'value_score']
].head(10))

# Worst value
print("\nTOP 10 WORST VALUE NEIGHBOURHOODS:")
print(value_analysis.sort_values('value_score', ascending=False)[
    ['neighbourhood_cleansed', 'avg_price', 'avg_rating', 'listing_count', 'value_score']
].tail(10))

TOP 10 BEST VALUE NEIGHBOURHOODS:
                  neighbourhood_cleansed  avg_price  avg_rating  \
89                                 Ost 2      58.13        4.68   
90                        Ostpreußendamm      71.96        4.88   
8                           Altglienicke      79.52        4.87   
13   Blankenburg/Heinersdorf/Märchenland      79.11        4.66   
105                                Rudow      82.52        4.74   
64                           Lichtenrade      87.38        4.88   
15                             Bohnsdorf      90.17        4.81   
12                              Biesdorf      95.04        4.86   
94                           Parkviertel      91.08        4.61   
11                        Baumschulenweg      96.00        4.82   

     listing_count  value_score  
89              31        8.051  
90              26        6.782  
8               23        6.124  
13              27        5.891  
105             46        5.744  
64              21      

In [7]:
# Price by number of people it accommodates
accommodates_analysis = df.groupby('accommodates').agg(
    avg_price=('price', 'mean'),
    median_price=('price', 'median'),
    count=('id', 'count')
).round(2)

# Only show groups with meaningful sample size
print(accommodates_analysis[accommodates_analysis['count'] >= 20])

              avg_price  median_price  count
accommodates                                
1                 68.09          58.0    791
2                 98.38          85.0   3910
3                116.95          97.0   1038
4                151.26         133.0   1812
5                177.26         163.0    439
6                199.99         172.0    640
7                207.46         194.0    138
8                247.91         227.0    203
9                251.26         233.5     66
10               277.45         237.0     65
11               274.00         277.0     27
12               321.24         298.0     37


In [8]:
# Correlation of numeric columns with price
numeric_cols = ['price', 'accommodates', 'bedrooms', 'beds', 
                'bathrooms', 'number_of_reviews', 'review_scores_rating',
                'availability_365', 'calculated_host_listings_count']

correlation = df[numeric_cols].corr()['price'].sort_values(ascending=False)
print(correlation)

price                             1.000000
accommodates                      0.530590
bedrooms                          0.474237
beds                              0.403896
bathrooms                         0.308700
availability_365                  0.067518
review_scores_rating              0.036485
calculated_host_listings_count    0.011682
number_of_reviews                -0.005720
Name: price, dtype: float64


In [9]:
# Save all analysis results
room_type_analysis.to_csv('/Users/ilakia/berlin-rental-analysis/processed/room_type_analysis.csv')
superhost_analysis.to_csv('/Users/ilakia/berlin-rental-analysis/processed/superhost_analysis.csv')
value_analysis.to_csv('/Users/ilakia/berlin-rental-analysis/processed/value_analysis.csv')
accommodates_analysis.to_csv('/Users/ilakia/berlin-rental-analysis/processed/accommodates_analysis.csv')

print("All analysis files saved!")

All analysis files saved!
